In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import lightgbm as lgb
import xgboost as xgb

import eda_util
import time
import json

import torch
from torch import nn
from torch import optim
import torch.utils.data as data_utils
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {DEVICE} device")

Using cuda device


In [3]:
RANDOM_STATE = 1048576
TARGET = 'accident_risk'

# Data Loading

In [4]:
train_csv = pd.read_csv('datasets/train_split.csv', index_col=0)
val_csv = pd.read_csv('datasets/val_split.csv', index_col=0)
test_csv = pd.read_csv('datasets/test_split.csv', index_col=0)

train_csv = pd.concat([train_csv, val_csv]).reset_index(drop=True)
X_train, y_train = train_csv.drop(columns=[TARGET]), train_csv[TARGET]
X_test, y_test = test_csv.drop(columns=[TARGET]), test_csv[TARGET]
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')

X_train shape: (465978, 12), X_test shape: (51776, 12)


In [5]:
def create_data_loader(X: pd.DataFrame | np.ndarray, y: pd.Series | np.ndarray,
                       batch_size: int = 32, num_workers: int = 0, persistent_workers: bool=False, shuffle: bool = True):
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    data_tensor = data_utils.TensorDataset(X, y)
    data_loader = data_utils.DataLoader(dataset=data_tensor,
                                        batch_size=batch_size,
                                        num_workers=num_workers,
                                        shuffle=shuffle,
                                        persistent_workers=persistent_workers)
    return data_loader

# Preprocessing

In [6]:
ord_features = ['lighting', 'time_of_day']
nom_features = ['road_type', 'weather']
cat_features = ord_features + nom_features
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']
num_features = X_train.columns.difference(cat_features + bool_features).tolist()

## Categorical Encoding

In [7]:
lgb_encoders = ColumnTransformer([
    ('nom_features', OrdinalEncoder(dtype=float, handle_unknown='use_encoded_value', unknown_value=np.nan), nom_features),
    ('ord_features', OrdinalEncoder(dtype=float, categories=[
        ['night', 'dim', 'daylight'], 
        ['morning', 'afternoon', 'evening']
        ], handle_unknown='use_encoded_value', unknown_value=np.nan), ord_features),
    ('bool_features', OrdinalEncoder(dtype=float, categories=[
        [False, True] for i in range(len(bool_features))
        ], handle_unknown='use_encoded_value', unknown_value=np.nan), bool_features)
], 
remainder='passthrough',
verbose_feature_names_out=False)
lgb_encoders.set_output(transform='pandas')
lgb_encoders

,transformers,"[('nom_features', ...), ('ord_features', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,dtype,<class 'float'>
,handle_unknown,'use_encoded_value'


In [8]:
xgb_encoders = ColumnTransformer([
    ('nom_features', OrdinalEncoder(dtype=float, handle_unknown='use_encoded_value', unknown_value=np.nan), nom_features),
    ('ord_features', OrdinalEncoder(dtype=float, categories=[
        ['night', 'dim', 'daylight'], 
        ['morning', 'afternoon', 'evening']
        ], handle_unknown='use_encoded_value', unknown_value=np.nan), ord_features),
    ('bool_features', OrdinalEncoder(dtype=float, categories=[
        [False, True] for i in range(len(bool_features))
        ], handle_unknown='use_encoded_value', unknown_value=np.nan), bool_features)
], 
remainder='passthrough',
verbose_feature_names_out=False)
xgb_encoders.set_output(transform='pandas')
xgb_encoders

,transformers,"[('nom_features', ...), ('ord_features', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,dtype,<class 'float'>
,handle_unknown,'use_encoded_value'


In [9]:
nn_encoders = ColumnTransformer([
    ('nom_features', OneHotEncoder(dtype=float, sparse_output=False), nom_features),
    ('ord_features', OneHotEncoder(dtype=float, sparse_output=False), ord_features),
    ('bool_features', OneHotEncoder(dtype=float, sparse_output=False), bool_features)
], 
remainder='passthrough',
verbose_feature_names_out=False)
nn_encoders.set_output(transform='pandas')
nn_encoders

,transformers,"[('nom_features', ...), ('ord_features', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [10]:
new_features = ['curvature_speed_limit', 'curvature_speed_limit_2']

def to_category(X):
    X = X.copy()
    for col in X.columns:
        if col not in num_features + new_features:
            X[col] = X[col].astype('category')
    return X

category_formatter = FunctionTransformer(to_category)
category_formatter

,func,<function to_...001DCA0004AE0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None
,inv_kw_args,None


## Feature Engineering

In [11]:
def add_related_features(X):
    X = X.copy()
    X['curvature_speed_limit'] = X['curvature'] * X['speed_limit']
    X['curvature_speed_limit_2'] = X['speed_limit'] ** 2 * X['curvature']
    return X

add_transformer = FunctionTransformer(add_related_features)
add_transformer

,func,<function add...001DCA0004A40>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None
,inv_kw_args,None


## Preprocessing Pipeline

In [12]:
def build_full_preprocessor(model: str):
    if model == 'lgb' or model == 'xgb':
        full_preprocessor = Pipeline(steps=[
            ('encoder', lgb_encoders),
            ('add_rel_feature', add_transformer)
        ])
    elif model == 'xgb':
        full_preprocessor = Pipeline(steps=[
            ('encoder', xgb_encoders),
            ('add_rel_feature', add_transformer),
            ('to_category', category_formatter)
        ])
    elif model == 'nn':
        full_preprocessor = Pipeline(steps=[
            ('encoder', nn_encoders),
            ('add_rel_feature', add_transformer),
            ('scaling', StandardScaler())
        ])
    else:
        raise ValueError("Model must be one of 'lgb', 'xgb', or 'nn'.")
    
    return full_preprocessor

# Model building

## Base Models

### Best params

In [13]:
models_name = ['lgb', 'xgb', 'nn']
best_params = {}
for name in models_name:
    best_params[name] = json.load(open(f'models/{name}_optuna_best_params.json', 'r'))
best_params

{'lgb': {'max_depth': 12,
  'num_leaves': 130,
  'min_child_samples': 39,
  'colsample_bytree': 0.8489781878210303,
  'subsample': 0.8706695817878217,
  'subsample_freq': 13,
  'reg_alpha': 0.17073749483459966,
  'reg_lambda': 0.14838906044340966,
  'min_split_gain': 0.0014250082982406012},
 'xgb': {'max_depth': 7,
  'num_leaves': 41,
  'min_child_samples': 48,
  'colsample_bytree': 0.9749961115071797,
  'subsample': 0.9996537505319927,
  'subsample_freq': 46,
  'reg_alpha': 0.7821288562799138,
  'reg_lambda': 0.5578363978627132,
  'min_split_gain': 0.41223227482885366},
 'nn': {'n_layers': 2,
  'n_units_l0': 128,
  'n_units_l1': 32,
  'dropout': 0.1,
  'learning_rate': 0.004474601474071384,
  'weight_decay': 7.620608670519667e-05}}

In [14]:
# Define the neural network model
class BaseNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int]=[128, 64, 32], dropout: float=0.2):
        """
        Fully Connected Neural Network for Regression tasks.

        Args:
            input_dim (int): Number of input features after preprocessing/encoding.
            hidden_dims (list of int): Hidden layer sizes.
            dropout (float): Dropout probability for regularization.
        """
        super(BaseNN, self).__init__()

        # Define the layers
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, X: pd.DataFrame | np.ndarray) -> pd.DataFrame | np.ndarray:
        '''
        Forward pass through the network.
        '''
        return self.model(X).squeeze(1)



In [15]:
def create_base_models(models_name: list[str]=models_name, best_params: dict=best_params, verbose: int=1):
    base_models = {}
    for name in models_name:
        if name == 'lgb':
            lgb_verbosity = verbose if verbose > 0 else -1
            model = LGBMRegressor(**best_params[name], random_state=RANDOM_STATE, verbosity=lgb_verbosity)
        elif name == 'xgb':
            model = XGBRegressor(**best_params[name], random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='rmse', verbosity=verbose)
        elif name == 'nn':
            hidden_dims = []
            for layer in range(best_params[name]['n_layers']):
                hidden_dims.append(best_params[name][f'n_units_l{layer}'])
            input_dim = build_full_preprocessor('nn').fit_transform(X_train).shape[1]
            model = BaseNN(
                input_dim=input_dim, 
                hidden_dims=hidden_dims, 
                dropout=best_params[name]['dropout']
            )
            model.to(DEVICE)
        else:
            raise ValueError("Model must be one of 'lgb', 'xgb', or 'nn'.")
        
        base_models[name] = model
    return base_models

In [16]:
def base_train_loop(model: BaseNN, 
               train_loader: data_utils.DataLoader, 
               val_loader: data_utils.DataLoader,
               criterion: nn.Module, 
               optimizer: optim.Optimizer, 
               scheduler: optim.lr_scheduler._LRScheduler,
               num_epochs: int = 100, 
               patience: int = 10,
               verbose: int = 1) -> tuple[float, float]:
    """
    Train the neural network model with early stopping.

    Args:
        model (RegressionNN): The neural network model to train.
        train_loader (DataLoader): DataLoader for training data.
        val_loader (DataLoader): DataLoader for validation data.
        criterion (nn.Module): Loss function.
        optimizer (optim.Optimizer): Optimizer for training.
        scheduler (optim.lr_scheduler._LRScheduler): Learning rate scheduler.
        num_epochs (int): Maximum number of epochs to train.
        patience_limit (int): Number of epochs with no improvement to wait before stopping.

    Returns:
        tuple: Best validation loss and corresponding RMSE score.
    """
    epoch_durations = []
    best_val_loss = float('inf')
    epoch_since = 0
    for epoch in range(num_epochs):
        start_time = time.time()

        model.train()
        train_loss = 0.0
        train_score = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)
            train_score += rmse(y_batch.cpu().detach().numpy(), outputs.cpu().detach().numpy()) * X_batch.size(0)

        train_loss /= len(train_loader.dataset)
        train_score /= len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_score = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_score += rmse(y_batch.cpu(), outputs.cpu()) * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_score /= len(val_loader.dataset)

        end_time = time.time()
        epoch_duration = end_time - start_time
        epoch_durations.append(epoch_duration)
        mins, secs = divmod(epoch_duration, 60)

        scheduler.step(val_loss)

        if verbose > 1:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Score: {train_score:.4f}, Val Loss: {val_loss:.4f}, Val Score: {val_score:.4f}, Time: {int(mins)}m {secs:.1f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "models/best_nn_model.pt")
            epoch_since = 0
        else:
            epoch_since += 1
            if epoch_since >= patience:
                if verbose > 0:
                    print("Early stopping triggered")
                break
    if verbose > 0:
        print(f'Average epoch duration: {np.mean(epoch_durations):.2f} seconds, Total training time: {np.sum(epoch_durations)//60} minutes {np.sum(epoch_durations) % 60:.2f} seconds, Best Val Score: {np.sqrt(best_val_loss):.4f}')
    return best_val_loss, np.sqrt(best_val_loss)

In [17]:
def generate_oof_predictions(models_name, X_train, y_train, X_test, best_params, verbose=1):
    kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros((len(X_train), len(models_name)))
    test_preds = np.zeros((len(X_test), len(models_name)))
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        # Creating models
        base_models = create_base_models(models_name=models_name, best_params=best_params, verbose=verbose)

        preprocessors = {}

        for i, name in enumerate(models_name):
            # Preprocessing
            preprocess_pipeline = build_full_preprocessor(model=name)
            preprocessors[name] = preprocess_pipeline
            X_tr_preprocessed = preprocess_pipeline.fit_transform(X_tr)
            X_val_preprocessed = preprocess_pipeline.transform(X_val)

            # Training and prediction
            if name != 'nn':
                base_models[name].fit(X_tr_preprocessed, y_tr)
                oof_preds[val_idx, i] = base_models[name].predict(X_val_preprocessed)
                test_preds[:, i] += base_models[name].predict(preprocess_pipeline.transform(X_test)) / kf.n_splits
            else:
                tr_loader = create_data_loader(X_tr_preprocessed, y_tr, batch_size=2048, num_workers=6, persistent_workers=True)
                val_loader = create_data_loader(X_val_preprocessed, y_val, batch_size=2048, num_workers=6, persistent_workers=True, shuffle=False)
                criterion = nn.MSELoss()
                optimizer = torch.optim.Adam(base_models[name].parameters(), lr=best_params['nn']['learning_rate'], weight_decay=best_params['nn']['weight_decay'])
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=1e-4)
                best_val_loss, best_val_score = base_train_loop(base_models[name], tr_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=10, verbose=1)
                base_models[name].eval()
                with torch.no_grad():
                    oof_preds[val_idx, i] = base_models[name](torch.tensor(preprocess_pipeline.transform(X_val), dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()
                    test_preds[:, i] += base_models[name](torch.tensor(preprocess_pipeline.transform(X_test), dtype=torch.float32).to(DEVICE)).cpu().detach().numpy() / kf.n_splits
        
        print(f'Completed fold {fold + 1}/{kf.n_splits}')
        
    return oof_preds, test_preds, base_models, preprocessors


In [18]:
oof_preds, test_preds, base_models, preprocessors = generate_oof_predictions(models_name, X_train, y_train, X_test, best_params, verbose=0)

Early stopping triggered
Average epoch duration: 1.49 seconds, Total training time: 1.0 minutes 41.29 seconds, Best Val Score: 0.0570
Completed fold 1/10
Early stopping triggered
Average epoch duration: 1.68 seconds, Total training time: 0.0 minutes 30.23 seconds, Best Val Score: 0.0577
Completed fold 2/10
Early stopping triggered
Average epoch duration: 1.49 seconds, Total training time: 1.0 minutes 59.06 seconds, Best Val Score: 0.0564
Completed fold 3/10
Early stopping triggered
Average epoch duration: 1.51 seconds, Total training time: 2.0 minutes 12.53 seconds, Best Val Score: 0.0564
Completed fold 4/10
Average epoch duration: 1.43 seconds, Total training time: 2.0 minutes 23.49 seconds, Best Val Score: 0.0564
Completed fold 5/10
Early stopping triggered
Average epoch duration: 1.48 seconds, Total training time: 1.0 minutes 33.11 seconds, Best Val Score: 0.0571
Completed fold 6/10
Early stopping triggered
Average epoch duration: 1.43 seconds, Total training time: 2.0 minutes 13.40

## Meta Model

In [19]:
class MetaNN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=8, dropout=0.1):
        super().__init__()
        # Define the layers
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden_dim, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, X: pd.DataFrame | np.ndarray) -> pd.DataFrame | np.ndarray:
        '''
        Forward pass through the network.
        '''
        return self.model(X).squeeze(1)

In [20]:
def meta_train_loop(model: BaseNN, 
               oof_pred: np.ndarray,
               y_train: pd.Series,
               criterion: nn.Module, 
               optimizer: optim.Optimizer, 
               scheduler: optim.lr_scheduler._LRScheduler,
               batch_size: int = 2048,
               num_workers: int = 6,
               persistent_workers: bool = True,
               n_splits: int = 5,
               num_epochs: int = 100, 
               patience: int = 10,
               verbose: int = 1) -> tuple[float, float]:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    val_scores = []
    fold_durations = []
    best_val_loss = float('inf')

    # Split data
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        fold_start_time = time.time()
        X_tr, y_tr = oof_pred[tr_idx], y_train.iloc[tr_idx]
        X_val, y_val = oof_pred[val_idx], y_train.iloc[val_idx]
        tr_loader = create_data_loader(X_tr, y_tr,
                                            batch_size=batch_size, 
                                            shuffle=True, 
                                            num_workers=num_workers, 
                                            persistent_workers=persistent_workers)
        val_loader = create_data_loader(X_val, y_val,
                                            batch_size=batch_size, 
                                            shuffle=False, 
                                            num_workers=num_workers, 
                                            persistent_workers=persistent_workers)

        
        epoch_durations = []
        eopch_since = 0
        for epoch in range(num_epochs):
            epoch_start_time = time.time()

            # Training Phase
            model.train()
            train_loss = 0.0
            train_score = 0.0
            for X_batch, y_batch in tr_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * X_batch.size(0)
                train_score += rmse(y_batch.cpu().detach().numpy(), outputs.cpu().detach().numpy()) * X_batch.size(0)
            
            train_loss /= len(tr_loader.dataset)
            train_score /= len(tr_loader.dataset)

            # Validation Phase
            model.eval()
            val_loss = 0.0
            val_score = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item() * X_batch.size(0)
                    val_score += rmse(y_batch.cpu(), outputs.cpu()) * X_batch.size(0)

            val_loss /= len(val_loader.dataset)
            val_score /= len(val_loader.dataset)

            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - epoch_start_time
            epoch_durations.append(epoch_duration)
            mins, secs = divmod(epoch_duration, 60)

            scheduler.step(val_loss)

            if verbose > 1:
                print(f'Fold {fold+1}/{n_splits}, Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Score: {train_score:.4f}, Val Loss: {val_loss:.4f}, Val Score: {val_score:.4f}, Time: {int(mins)}m {secs:.1f}s')

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epoch_since = 0
            else:
                epoch_since += 1
                if epoch_since >= patience:
                    if verbose > 0:
                        print("Early stopping triggered")
                    break
        fold_end_time = time.time()
        fold_duration = fold_end_time - fold_start_time
        fold_durations.append(fold_duration)
        if verbose > 0:
            print(f'Fold {fold+1} Average epoch duration: {np.mean(epoch_durations):.2f} seconds, Fold training time: {fold_duration//60} minutes {fold_duration % 60:.2f} seconds, Best Val Score: {np.sqrt(best_val_loss):.4f}')
        val_scores.append(np.sqrt(best_val_loss))
    return np.mean(val_scores), np.std(val_scores)       

In [21]:
meta_model = MetaNN(input_dim=len(models_name), hidden_dim=8, dropout=0.1)
meta_model.to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(meta_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=1e-4)
mean_val_score, std_val_score = meta_train_loop(
    meta_model,
    oof_preds,
    y_train,
    criterion,
    optimizer,
    scheduler, 
    batch_size=2048,
    num_workers=6,
    persistent_workers=True,
    n_splits=10,
    num_epochs=100,
    patience=10,
    verbose=1
)

Early stopping triggered
Fold 1 Average epoch duration: 1.33 seconds, Fold training time: 0.0 minutes 42.59 seconds, Best Val Score: 0.0566
Early stopping triggered
Fold 2 Average epoch duration: 7.16 seconds, Fold training time: 0.0 minutes 8.30 seconds, Best Val Score: 0.0566
Early stopping triggered
Fold 3 Average epoch duration: 1.35 seconds, Fold training time: 0.0 minutes 41.68 seconds, Best Val Score: 0.0560
Early stopping triggered
Fold 4 Average epoch duration: 7.90 seconds, Fold training time: 0.0 minutes 9.08 seconds, Best Val Score: 0.0560
Early stopping triggered
Fold 5 Average epoch duration: 6.76 seconds, Fold training time: 0.0 minutes 7.92 seconds, Best Val Score: 0.0560
Early stopping triggered
Fold 6 Average epoch duration: 7.18 seconds, Fold training time: 0.0 minutes 8.35 seconds, Best Val Score: 0.0560
Early stopping triggered
Fold 7 Average epoch duration: 6.55 seconds, Fold training time: 0.0 minutes 7.65 seconds, Best Val Score: 0.0560
Early stopping triggered


In [22]:
print(f'Meta-Model OOF CV RMSE: {mean_val_score:.4f} ± {std_val_score:.4f}')

Meta-Model OOF CV RMSE: 0.0560 ± 0.0003


In [23]:
print(f'Meta-Model Test RMSE: {rmse(y_test, meta_model(torch.tensor(test_preds, dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()):.4f}')

Meta-Model Test RMSE: 0.0566


# Prediction and Submission


In [24]:
def full_predict(meta_model: nn.Module, 
                 base_models: dict[str, any], 
                 full_preprocessor: dict[str, Pipeline],
                 X: pd.DataFrame) -> np.ndarray:
    """
    Generate predictions using the meta-model and base models.

    Args:
        meta_model (nn.Module): The trained meta-model.
        base_models (dict): Dictionary of trained base models.
        X (pd.DataFrame): Input features for prediction.

    Returns:
        np.ndarray: Predictions from the meta-model.
    """
    oof_pred = np.zeros((len(X), len(base_models)))
    for i, name in enumerate(base_models.keys()):
        X_preprocessed = full_preprocessor[name].transform(X)

        if name != 'nn':
            oof_pred[:, i] = base_models[name].predict(X_preprocessed)
        else:
            base_models[name].eval()
            with torch.no_grad():
                oof_pred[:, i] = base_models[name](torch.tensor(X_preprocessed, dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()
    
    meta_model.eval()
    with torch.no_grad():
        predictions = meta_model(torch.tensor(oof_pred, dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()
    
    return predictions

In [28]:
full_test = pd.read_csv('datasets/test.csv', index_col=0)
y_pred = full_predict(meta_model, base_models, preprocessors, full_test)
submission = pd.read_csv('datasets/sample_submission.csv', index_col=0)
print(submission.head())
submission[TARGET] = y_pred.round(3)
print('\n')
print(submission.head())
submission.to_csv('datasets/submission_stacking.csv')
print('Result saved successfully!')

        accident_risk
id                   
517754          0.352
517755          0.352
517756          0.352
517757          0.352
517758          0.352


        accident_risk
id                   
517754          0.299
517755          0.128
517756          0.183
517757          0.318
517758          0.407
Result saved successfully!
